# DeepSketch Phase 2 Training on Kaggle

This notebook orchestrates the existing Phase 2 scripts without duplicating training logic.

Pipeline:
1. Clone the repository
2. Install dependencies
3. Link the paired sketch dataset
4. Generate deterministic stylized targets
5. Train the Phase 2 Pix2Pix model
6. Preview outputs and package artifacts

In [ ]:
import os
import shutil
from pathlib import Path

os.environ['WANDB_DISABLED'] = 'true'
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES', 'not set'))
!nvidia-smi
!python --version

In [ ]:
%cd /kaggle/working
repo_dir = Path('/kaggle/working/deep-sketch')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone https://github.com/<YOUR_GITHUB_USERNAME>/deep-sketch.git
%cd /kaggle/working/deep-sketch

Replace `<YOUR_GITHUB_USERNAME>` in the clone command above before running the notebook.

In [ ]:
!pip install -r requirements.txt

In [ ]:
kaggle_input_root = Path('/kaggle/input/cufs-dataset-clean')
photos_src = kaggle_input_root / 'photos'
sketches_src = kaggle_input_root / 'sketches'
print('photos exists:', photos_src.exists())
print('sketches exists:', sketches_src.exists())
if not photos_src.exists() or not sketches_src.exists():
    raise FileNotFoundError('Expected /kaggle/input/cufs-dataset-clean/{photos,sketches}')

dataset_root = Path('dataset')
dataset_root.mkdir(exist_ok=True)
for target_name, source_path in [('photos', photos_src), ('sketches', sketches_src)]:
    target_path = dataset_root / target_name
    if target_path.exists() or target_path.is_symlink():
        if target_path.is_symlink() or target_path.is_file():
            target_path.unlink()
        else:
            shutil.rmtree(target_path)
    os.symlink(source_path, target_path, target_is_directory=True)

print('Linked dataset directories:')
!ls -la dataset

In [ ]:
!python scripts/prepare_stylized_data.py --input-dir dataset/photos --output-dir dataset/stylized --points 1200 --colors 12 --seed 42

In [ ]:
!python phase2/training/train.py --epochs 200

In [ ]:
from IPython.display import Image, display
from pathlib import Path

sample_dir = Path('samples/phase2')
sample_files = sorted(sample_dir.glob('*.png'))
print('num sample files:', len(sample_files))
if sample_files:
    display(Image(filename=str(sample_files[-1])))
else:
    print('No sample grids found yet.')

In [ ]:
!python phase2/inference/generate_color.py --input dataset/sketches/$(ls dataset/sketches | head -n 1) --output phase2_preview.png
!python - <<'PY'
from IPython.display import Image, display
display(Image(filename='phase2_preview.png'))
PY

In [ ]:
!zip -r phase2_artifacts.zip phase2/checkpoints samples/phase2 phase2_preview.png
print('Created phase2_artifacts.zip')